In [1]:
!pip install pandas textblob symspellpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.3/168.3 kB 4.8 MB/s eta 0:00:00


In [2]:
!pip install -U transformers torch tqdm pandas


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 877.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 76.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.

In [3]:
# 3. Create the Kaggle directory and move the file there
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
# Change permissions so no one else can read your API key
!chmod 600 ~/.kaggle/kaggle.json

# 4. Download the Amazon Fine Food Reviews dataset
print("\nDownloading the Amazon dataset...")
!kaggle datasets download -d snap/amazon-fine-food-reviews

# 5. Unzip the downloaded file
print("Unzipping the data...")
!unzip -q amazon-fine-food-reviews.zip

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory

Dataset URL: https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews
License(s): CC0-1.0
100% 242M/242M [00:02<00:00, 118MB/s] 

Unzipping the data...


In [4]:
import pandas as pd
from transformers import pipeline
from tqdm.notebook import tqdm

# Enable a progress bar for pandas (useful so you know it hasn't frozen!)
tqdm.pandas()

# 1. Load the dataset you collected in the previous step
df_hn = pd.read_csv("/content/Reviews.csv")
df_hn=df_hn.loc[0:5,:]


print(df_hn.shape)

(6, 10)


In [7]:
df_hn.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


regex data cleaning

In [11]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords

# 1. Download the standard English stopwords list
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))

# 2. Define the Advanced Regex Cleaning Function
def advanced_regex_clean(text):
    if not isinstance(text, str):
        return ""

    # Remove leftover URLs (http/https or www)
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # Remove punctuation and special characters (Keep only letters and spaces)
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Remove multiple spaces caused by deletions and strip leading/trailing whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # Convert to lowercase to ensure stopword matching works perfectly
    return text.lower()

# 3. Define the Stopword Removal Function
def remove_stopwords(text):
    if not isinstance(text, str):
        return ""

    # Split text into a list of words, filter out stopwords, and rejoin into a string
    clean_words = [word for word in text.split() if word not in stop_words]
    return " ".join(clean_words)

# 4. Define the Regex Categorization Function (Fulfills your Task 2 Rubric)
def categorize_by_url(url):
    if pd.isna(url) or not isinstance(url, str):
        return "General Discussion"

    # Use Regex to search for specific domains within the Story_URL
    if re.search(r'github\.com', url, re.IGNORECASE):
        return "Open Source / Code"
    elif re.search(r'nytimes\.com|wsj\.com|bloomberg\.com', url, re.IGNORECASE):
        return "Mainstream News"
    elif re.search(r'arxiv\.org|nature\.com', url, re.IGNORECASE):
        return "Academic Research"
    elif re.search(r'youtube\.com', url, re.IGNORECASE):
        return "Video Media"
    else:
        return "Tech Blog / Other"

# ==========================================
# EXECUTE THE PIPELINE
# ==========================================

print("Loading dataset...")
df_hn = pd.read_csv("/content/Reviews.csv")

print("Applying Regex Categorization Tags...")

print("Applying Advanced Regex Cleaning...")
# We use the initial 'Text' column as the base
df_hn['Regex_Cleaned_Text'] = df_hn['Text'].apply(advanced_regex_clean)

print("Removing Stopwords...")
# We create a brand new column from the regex-cleaned text
df_hn['Text_Without_Stopwords'] = df_hn['Regex_Cleaned_Text'].apply(remove_stopwords)

# Save the final preprocessed dataset
output_file = "hn_fully_preprocessed_data.csv"
df_hn.to_csv(output_file, index=False)

print(f"\n✅ Processing Complete! Saved to: {output_file}")
print("-" * 50)
print("Preview of the new columns:")
display(df_hn[['Text', 'Regex_Cleaned_Text', 'Text_Without_Stopwords']].head(5))

Loading dataset...
Applying Regex Categorization Tags...
Applying Advanced Regex Cleaning...
Removing Stopwords...

✅ Processing Complete! Saved to: hn_fully_preprocessed_data.csv
--------------------------------------------------
Preview of the new columns:


,Text,Regex_Cleaned_Text,Text_Without_Stopwords
0,I have bought several of the Vitality canned d...,i have bought several of the vitality canned d...,bought several vitality canned dog food produc...
1,Product arrived labeled as Jumbo Salted Peanut...,product arrived labeled as jumbo salted peanut...,product arrived labeled jumbo salted peanutsth...
2,This is a confection that has been around a fe...,this is a confection that has been around a fe...,confection around centuries light pillowy citr...
3,If you are looking for the secret ingredient i...,if you are looking for the secret ingredient i...,looking secret ingredient robitussin believe f...
4,Great taffy at a great price. There was a wid...,great taffy at a great price there was a wide ...,great taffy great price wide assortment yummy ...


assigns a tag based on what it finds:

Keyword github.com ➔ Gets tagged as "Open Source / Code"
(Because if the link goes to GitHub, the discussion is likely about programming or a new open-source AI model).

Keywords nytimes.com, wsj.com, or bloomberg.com ➔ Gets tagged as "Mainstream News"
(Because these are major news outlets, meaning the discussion is likely about the business, politics, or public impact of AI).

Keywords arxiv.org or nature.com ➔ Gets tagged as "Academic Research"
(Because arXiv is where scientific AI papers are published).

Keyword youtube.com ➔ Gets tagged as "Video Media"

Fallback: If the URL doesn't match any of those, or if it's just a text post with no link, it defaults to "Tech Blog / Other" or "General Discussion".

In [ ]:
df_hn["Category_Tag"].value_counts()

Spell corection

In [12]:
import pandas as pd
from textblob import TextBlob
from symspellpy import SymSpell, Verbosity
import pkg_resources
import nltk

# Download the punkt_tab tokenizer data for TextBlob as suggested by the error
nltk.download('punkt_tab', quiet=True)
# Download the wordnet corpus for lemmatization
nltk.download('wordnet', quiet=True)

# 1. Initialize SymSpell (Spell Correction)
# Max edit distance 2 allows for 2 changes (insert/delete/replace) per word
sym_spell = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)
dictionary_path = pkg_resources.resource_filename(
    "symspellpy", "frequency_dictionary_en_82_765.txt"
)
sym_spell.load_dictionary(dictionary_path, term_index=0, count_index=1)

def apply_spell_correction(text):
    if not isinstance(text, str) or text == "":
        return text

    corrected_words = []
    for word in text.split():
        # Look for the best match in the dictionary
        suggestions = sym_spell.lookup(word, Verbosity.CLOSEST, max_edit_distance=2)
        corrected_words.append(suggestions[0].term if suggestions else word)
    return " ".join(corrected_words)

# 2. Define Lemmatization (TextBlob)
def apply_lemmatization(text):
    if not isinstance(text, str):
        return text
    # TextBlob identifies the root of the word (e.g., 'running' -> 'run')
    return " ".join([word.lemmatize() for word in TextBlob(text).words])

# --- APPLY TO DATAFRAME ---

print("Starting Spell Correction (SymSpell)...")
# We apply this to the regex cleaned text to ensure we aren't trying to fix URLs or symbols
df_hn['Spelling_Corrected_Text'] = df_hn['Regex_Cleaned_Text'].apply(apply_spell_correction)

print("Starting Lemmatization (TextBlob)...")
df_hn['Lemmatized_Text'] = df_hn['Spelling_Corrected_Text'].apply(apply_lemmatization)

# Save the final high-fidelity dataset
df_hn.to_csv("hn_final_refined_data.csv", index=False)

print("\n✅ Task 2 Complete! Data is cleaned, corrected, and lemmatized.")
display(df_hn[['Regex_Cleaned_Text', 'Spelling_Corrected_Text', 'Lemmatized_Text']].head())

/tmp/ipykernel_13936/3421086874.py:4: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


Starting Spell Correction (SymSpell)...
Starting Lemmatization (TextBlob)...

✅ Task 2 Complete! Data is cleaned, corrected, and lemmatized.


,Regex_Cleaned_Text,Spelling_Corrected_Text,Lemmatized_Text
0,i have bought several of the vitality canned d...,i have bought several of the vitality canned d...,i have bought several of the vitality canned d...
1,product arrived labeled as jumbo salted peanut...,product arrived labeled as jumbo salted peanut...,product arrived labeled a jumbo salted peanuts...
2,this is a confection that has been around a fe...,this is a confection that has been around a fe...,this is a confection that ha been around a few...
3,if you are looking for the secret ingredient i...,if you are looking for the secret ingredient i...,if you are looking for the secret ingredient i...
4,great taffy at a great price there was a wide ...,great taffy at a great price there was a wide ...,great taffy at a great price there wa a wide a...


In [13]:
df_hn.columns

Index(['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator',
       'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text',
       'Regex_Cleaned_Text', 'Text_Without_Stopwords',
       'Spelling_Corrected_Text', 'Lemmatized_Text'],
      dtype='str')